# ZarAI — QLoRA Fine-Tuning on Your Own Dataset (Llama-3.1-8B-Instruct) — Kaggle version

This is the Kaggle-adapted version of your QLoRA fine-tuning notebook. Same model, same training logic, same held-out evaluation — only the file upload/download and GPU setup steps are different, since Kaggle works differently from Colab in those areas.

**Before running, set up your Kaggle session:**
1. On the right-hand panel: **Settings → Accelerator → GPU T4 x2** (or P100)
2. **Settings → Internet → On** (required — you need internet access to download the model from Hugging Face and install packages)
3. **Add your dataset**: click **+ Add Input** (top right) → upload your Excel file as a new Kaggle Dataset, or select it if you've already uploaded it before
4. **Add your Hugging Face token as a Kaggle Secret** (recommended — avoids re-entering it every session): Add-ons → Secrets → add a secret named `HF_TOKEN` with your token as the value

**Kaggle session limits (free tier):** up to ~9 hours continuous runtime, ~30 GPU-hours/week quota — generally more generous than Colab's free tier, which is part of why this is worth moving here for a longer training run.

**You'll still need Hugging Face access to `meta-llama/Llama-3.1-8B-Instruct`** — same gated-model requirement as on Colab.

## 1. Install dependencies

In [1]:
!pip install -q -U transformers accelerate "bitsandbytes>=0.46.1" peft trl datasets pandas openpyxl sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 97.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 50.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 115.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency co

## 2. Check GPU

Make sure you enabled **Settings → Accelerator → GPU T4 x2** (or P100) as noted above before running this.

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 2b. Hugging Face login (via Kaggle Secrets)

Uses the `HF_TOKEN` secret you added in the setup step above — no need to paste your token into the notebook or log in interactively each session.

**If you haven't set up the secret yet:** Add-ons → Secrets (top menu) → add a secret named `HF_TOKEN` with your Hugging Face token as the value, then re-run this cell.

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to Hugging Face via Kaggle Secret.")

Logged in to Hugging Face via Kaggle Secret.


## 3. Load your dataset

On Kaggle, your dataset is attached via the **+ Add Input** panel (not uploaded inline like Colab). It appears under `/kaggle/input/<your-dataset-name>/`.

Run the cell below to list what's available, then set `DATA_PATH` to the correct file.

In [4]:
import os

# List available input datasets so you can find the exact path/filename
for root, dirs, filenames in os.walk("/kaggle/input"):
    for f in filenames:
        print(os.path.join(root, f))

# Update this to match your actual dataset file path from the list printed above
DATA_PATH = "/kaggle/input/datasets/mahmadghafoor/urdu-agriculture-question-asnwer/urdu- agriculture Question Asnwer.xlsx"
print("\nUsing DATA_PATH:", DATA_PATH)

/kaggle/input/datasets/mahmadghafoor/urdu-agriculture-question-asnwer/urdu- agriculture Question Asnwer.xlsx

Using DATA_PATH: /kaggle/input/datasets/mahmadghafoor/urdu-agriculture-question-asnwer/urdu- agriculture Question Asnwer.xlsx


In [5]:
import pandas as pd

df = pd.read_excel(DATA_PATH)
print("Shape:", df.shape)
df.head(3)

Shape: (1000, 13)


,id,question_urdu,question_roman_urdu,question_english,answer_urdu,answer_roman_urdu,answer_english,category,crop,season,region,difficulty_level,source_type
0,1,گندم میں پیلا زنگ کی بیماری کیسے پہچانی جائے؟,Gandum mein peela zang ki bimari kaise pehchan...,How to identify yellow rust disease in wheat?,گندم میں پیلا زنگ کی علامات: پتوں پر زرد دھاری...,Gandum mein peela zang ki alamaat: patton par ...,Yellow rust in wheat shows yellow stripes on l...,disease_management,wheat,rabi,Punjab,intermediate,expert_knowledge
1,2,گندم کو پانی کب دیں؟,Gandum ko paani kab dein?,When to irrigate wheat?,گندم کو 4 سنچائی چاہیں: کراؤن روٹ (21 دن)، کشت...,Gandum ko 4 senchai chahiyein: crown root (21 ...,Wheat needs 4 irrigations: crown root (21 days...,irrigation_management,wheat,rabi,Punjab,beginner,agricultural_extension
2,3,گندم کی بہترین اقسام کون سی ہیں؟,Gandum ki behtareen aqsaam konsi hain?,What are the best wheat varieties?,پنجاب میں NARC-2009، فیصل آباد-2008، اور پاکست...,Punjab mein NARC-2009; Faisalabad-2008; aur Pa...,In Punjab; NARC-2009; Faisalabad-2008; and Pak...,variety_selection,wheat,rabi,Punjab,beginner,agricultural_extension


## 4. Train / test split (held out BEFORE any training happens)

A stratified 85/15 split by `category`, so both sets have a representative
mix of question types. Uses `groupby().sample()` — not
`groupby().apply(lambda x: x.sample(...))`, since the `.apply()` version
silently drops the grouping column under pandas 3.0+.

`test_df` is set aside now and never touched during training — it's only
used in the final evaluation section.

In [6]:
TEST_FRACTION = 0.15
RANDOM_STATE = 42

test_df = (
    df.groupby("category", group_keys=False)
      .sample(frac=TEST_FRACTION, random_state=RANDOM_STATE)
)
train_df = df.drop(test_df.index).reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train rows: {len(train_df)}  |  Test rows (held out): {len(test_df)}")
print("\nTrain category distribution:")
print(train_df["category"].value_counts(normalize=True).round(2))
print("\nTest category distribution:")
print(test_df["category"].value_counts(normalize=True).round(2))

Train rows: 851  |  Test rows (held out): 149

Train category distribution:
category
pest_management               0.15
disease_management            0.14
nutrient_deficiency           0.07
irrigation_management         0.07
post_harvest                  0.06
fertilizer_management         0.05
weed_management               0.05
organic_farming               0.04
modern_technology             0.04
yield_improvement             0.03
crop_establishment            0.03
climate_adaptation            0.02
soil_management               0.02
seed_treatment                0.02
planting_calendar             0.02
water_management              0.02
variety_selection             0.02
organic_pest_management       0.01
soil_fertility                0.01
crop_physiology               0.01
integrated_pest_management    0.01
agricultural_challenges       0.01
nursery_management            0.01
soil_health                   0.01
land_preparation              0.01
water_stress                  0.01
crop_

## 5. Build training examples

One training example per (row, language) pair — so a single dataset row with
all three languages filled in produces up to 3 training examples. This
teaches the model all three languages from the same underlying training set,
consistent with how you've been evaluating it.

In [7]:
SYSTEM_PROMPT = "You are a helpful agricultural advisor for farmers in Pakistan. Answer clearly and practically."

lang_cols = {
    "urdu": ("question_urdu", "answer_urdu"),
    "roman_urdu": ("question_roman_urdu", "answer_roman_urdu"),
    "english": ("question_english", "answer_english"),
}

def build_training_examples(source_df):
    examples = []
    for _, row in source_df.iterrows():
        for lang, (qcol, acol) in lang_cols.items():
            q, a = row.get(qcol), row.get(acol)
            if pd.notna(q) and pd.notna(a) and str(q).strip() and str(a).strip():
                examples.append({
                    "language": lang,
                    "question": str(q).strip(),
                    "answer": str(a).strip(),
                })
    return examples

train_examples = build_training_examples(train_df)
print(f"Total training examples (row x language pairs): {len(train_examples)}")

Total training examples (row x language pairs): 2553


## 6. Load Llama-3.1-8B-Instruct in 4-bit (base for QLoRA)

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 256  # keeps sequences short to help fit training on a T4's 16GB

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,  # bf16 throughout avoids GradScaler/fp16 dtype conflicts entirely
    device_map="auto"
)
print("Base model loaded (4-bit):", MODEL_NAME)
print("GPU memory allocated: {:.2f} GB".format(torch.cuda.memory_allocated() / 1e9))

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Base model loaded (4-bit): meta-llama/Llama-3.1-8B-Instruct
GPU memory allocated: 1.50 GB


## 7. Format training examples using the chat template

In [9]:
from datasets import Dataset

def format_example(ex):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": ex["question"]},
        {"role": "assistant", "content": ex["answer"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

formatted = [format_example(ex) for ex in train_examples]
train_dataset = Dataset.from_list(formatted)
print(train_dataset)
print("\n--- Example formatted training text ---\n")
print(train_dataset[0]["text"])

Dataset({
    features: ['text'],
    num_rows: 2553
})

--- Example formatted training text ---

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful agricultural advisor for farmers in Pakistan. Answer clearly and practically.<|eot_id|><|start_header_id|>user<|end_header_id|>

گندم میں پیلا زنگ کی بیماری کیسے پہچانی جائے؟<|eot_id|><|start_header_id|>assistant<|end_header_id|>

گندم میں پیلا زنگ کی علامات: پتوں پر زرد دھاریاں اور پاؤڈر جیسا مادہ۔ متاثرہ پودوں کو نکالیں اور پروپیکونازول اسپرے کریں۔<|eot_id|>


## 8. Attach LoRA adapters (QLoRA setup)

In [10]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


## 9. Train

`NUM_EPOCHS = 3` is a reasonable starting point for ~2,500 examples on an 8B
model — enough to adapt without badly overfitting a dataset this size. Watch
the training loss: if it keeps dropping sharply through all 3 epochs with no
sign of leveling off, more epochs might help; if it flattens early, fewer
would avoid wasted compute.

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "zarai_llama31_qlora_adapter"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,      # small batch to fit T4 memory
    gradient_accumulation_steps=8,       # keeps effective batch size at 8
    gradient_checkpointing=True,         # trades some speed for a lot of memory savings
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,                           # bf16 needs no loss scaler, avoiding fp16/bf16 dtype conflicts
    optim="paged_adamw_8bit",
    report_to="none",
    dataset_text_field="text",
    loss_type="nll",  # falls back from chunked_nll, which errors on PEFT + gradient checkpointing setups (AttributeError: functools.partial has no __func__)
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()

In [ ]:
# Save the LoRA adapter (small — just the fine-tuned deltas, not the full 8B model)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter saved to:", OUTPUT_DIR)

# Zip and download so you don't lose it if the Colab session ends
import shutil
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
files.download(f"{OUTPUT_DIR}.zip")

## 10. Define the inference function (using the fine-tuned model)

In [ ]:
model.eval()

def ask_model(question, system_prompt=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,
            do_sample=True
        )
    generated = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# quick sanity check
print(ask_model("گندم کو پانی کب دیں؟"))

## 11. Load the SBERT scoring model (identical to your other notebooks)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print("Embedding model loaded:", EMBED_MODEL_NAME)

def semantic_similarity(text_a, text_b):
    emb = embed_model.encode([str(text_a), str(text_b)], normalize_embeddings=True)
    return float(np.dot(emb[0], emb[1])) * 100

## 12. Evaluate on the held-out test split (honest generalization score)

This is the critical difference from a normal evaluation run: **only
`test_df` is used here** — rows the fine-tuned model never saw during
training. This is what actually tells you whether fine-tuning improved
real performance, not just memorization.

Same SBERT scoring logic as every other model you've evaluated, so this
result is directly comparable to your Qwen/Llama/Qalb numbers.

In [ ]:
import time
from tqdm.auto import tqdm

LANGUAGES_TO_RUN = ["urdu", "roman_urdu", "english"]
CHECKPOINT_EVERY = 25
CHECKPOINT_PATH = "/kaggle/working/zarai_qlora_test_validation_checkpoint.xlsx"

full_results = []
start_time = time.time()
processed = 0

for lang in LANGUAGES_TO_RUN:
    qcol, acol = lang_cols[lang]
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc=f"Evaluating ({lang})"):
        question = row.get(qcol)
        reference_answer = row.get(acol)

        if pd.isna(question) or not str(question).strip():
            continue

        model_answer = ask_model(str(question))

        if pd.notna(reference_answer) and str(reference_answer).strip():
            accuracy_score = semantic_similarity(model_answer, reference_answer)
        else:
            accuracy_score = None

        full_results.append({
            "id": row.get("id"),
            "language": lang,
            "question": question,
            "model_answer": model_answer,
            "reference_answer": reference_answer,
            "semantic_accuracy_score": accuracy_score,
            "category": row.get("category"),
            "crop": row.get("crop"),
            "season": row.get("season"),
            "region": row.get("region"),
            "difficulty_level": row.get("difficulty_level"),
            "source_type": row.get("source_type"),
        })

        processed += 1
        if processed % CHECKPOINT_EVERY == 0:
            pd.DataFrame(full_results).to_excel(CHECKPOINT_PATH, index=False)

elapsed = time.time() - start_time
print(f"Done. Evaluated {len(full_results)} held-out question/answer pairs in {elapsed/60:.1f} minutes.")

results_df = pd.DataFrame(full_results)
results_df.head()

## 13. Aggregate stats + export full report

In [ ]:
def summarize(group_col):
    valid = results_df.dropna(subset=["semantic_accuracy_score"])
    return (
        valid.groupby(group_col)["semantic_accuracy_score"]
             .agg(["count", "mean", "median", "std", "min", "max"])
             .round(1)
             .sort_values("mean", ascending=False)
    )

overall = pd.DataFrame({
    "metric": ["total_evaluated", "scored_pairs", "mean_score", "median_score", "std_score"],
    "value": [
        len(results_df),
        results_df["semantic_accuracy_score"].notna().sum(),
        round(results_df["semantic_accuracy_score"].mean(), 1),
        round(results_df["semantic_accuracy_score"].median(), 1),
        round(results_df["semantic_accuracy_score"].std(), 1),
    ],
})

by_language = summarize("language")
by_category = summarize("category")
by_crop = summarize("crop")
by_difficulty = summarize("difficulty_level")

low_accuracy_flags = results_df[results_df["semantic_accuracy_score"] < 40]

print("Overall (held-out test set only):")
print(overall.to_string(index=False))
print("\nBy language:")
print(by_language)
print(f"\nRows with low semantic accuracy (<40/100): {len(low_accuracy_flags)}")

In [ ]:
FULL_REPORT_PATH = "/kaggle/working/zarai_qlora_llama31_test_validation.xlsx"

with pd.ExcelWriter(FULL_REPORT_PATH, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="test_set_results", index=False)
    overall.to_excel(writer, sheet_name="overall_summary", index=False)
    by_language.to_excel(writer, sheet_name="by_language")
    by_category.to_excel(writer, sheet_name="by_category")
    by_crop.to_excel(writer, sheet_name="by_crop")
    by_difficulty.to_excel(writer, sheet_name="by_difficulty")
    low_accuracy_flags.to_excel(writer, sheet_name="flag_low_accuracy", index=False)

print("Exported:", FULL_REPORT_PATH, "— available in the notebook's Output tab after this session.")